# Notebook 1: Scraping des matchs et boxscores V3 depuis 2010


In [2]:
from datetime import datetime

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate src/ directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from src.nba_scrapping import *
from src.utils import get_latest_file
from src.config import *
import pandas as pd

In [3]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-11-05 00:52:49.324080


# Saisons ciblées


In [4]:
#seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2011, 2015)]
#seasons = ["2000-01","2001-02","2002-03","2003-04","2004-5",]
seasons = ["2004-05","2003-04","2002-03","2001-02","2000-01",]
seasons


['2004-05', '2003-04', '2002-03', '2001-02', '2000-01']

In [5]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# 1. Récupération des matchs


In [7]:
path_games = download_games_for_seasons(seasons, DATA_GAMES_DIR, run_timestamp)


Extraction saison 2004-05
Extraction saison 2003-04
Extraction saison 2002-03
Extraction saison 2001-02
Extraction saison 2000-01


## run once : downloade teams

In [ ]:
*""" from nba_api.stats.static import teams


# Récupère toutes les équipes NBA
nba_teams = teams.get_teams()
teams_df = pd.DataFrame(nba_teams)

# # Garde les colonnes principales pour la lisibilité
main_cols = ['id', 'full_name', 'abbreviation', 'city', 'state', 'year_founded']
teams_df = teams_df[main_cols]












                    /*-
# # Affiche un aperçu
display(teams_df)



# # Sauvegarde en CSV
# #teams_df.to_csv('nba_teams.csv', index=False)

save_dataframe_to_csv(teams_df, DATA_TEAMS_DIR, prefix='nba_teams_', suffix=run_timestamp)
 """--

" from nba_api.stats.static import teams\n\n\n# Récupère toutes les équipes NBA\nnba_teams = teams.get_teams()\nteams_df = pd.DataFrame(nba_teams)\n\n# # Garde les colonnes principales pour la lisibilité\nmain_cols = ['id', 'full_name', 'abbreviation', 'city', 'state', 'year_founded']\nteams_df = teams_df[main_cols]\n\n# # Affiche un aperçu\ndisplay(teams_df)\n\n\n\n# # Sauvegarde en CSV\n# #teams_df.to_csv('nba_teams.csv', index=False)\n\nsave_dataframe_to_csv(teams_df, DATA_TEAMS_DIR, prefix='nba_teams_', suffix=run_timestamp)\n "

# 2. Chargement des GAME_IDs pour scraping boxscore


In [9]:



games_file = path_games #get_latest_file(DATA_GAMES_DIR)


print(f"Using games file: {games_file}")

games_df = pd.read_csv(games_file, dtype={'GAME_ID': str})
games_df


Using games file: /home/ju/Documents/Dev/NBA_Predictor/data/legacy/raw/games/2004-05_2000-01/nba_games_2025-11-05_00-52-49.csv


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22000,1.610613e+09,VAN,Vancouver Grizzlies,0020000013,2000-10-31,VAN vs. SEA,W,241,94,...,14,37,51,28,12,6,11,21,6.0,2000-01
1,22000,1.610613e+09,IND,Indiana Pacers,0020000009,2000-10-31,IND @ SAS,L,240,85,...,9,26,35,19,5,4,16,32,-13.0,2000-01
2,22000,1.610613e+09,POR,Portland Trail Blazers,0020000012,2000-10-31,POR vs. LAL,L,239,86,...,13,19,32,18,13,1,10,28,-10.0,2000-01
3,22000,1.610613e+09,PHX,Phoenix Suns,0020000011,2000-10-31,PHX @ GSW,L,240,94,...,11,33,44,25,12,3,16,28,-2.0,2000-01
4,22000,1.610613e+09,CLE,Cleveland Cavaliers,0020000002,2000-10-31,CLE @ NJN,W,240,86,...,11,41,52,16,5,8,19,27,4.0,2000-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12759,42004,1.610613e+09,SAS,San Antonio Spurs,0040400405,2005-06-19,SAS @ DET,W,265,96,...,19,26,45,20,3,3,16,26,1.0,2004-05
12760,42004,1.610613e+09,SAS,San Antonio Spurs,0040400406,2005-06-21,SAS vs. DET,L,240,86,...,13,30,43,15,3,2,11,18,-9.0,2004-05
12761,42004,1.610613e+09,DET,Detroit Pistons,0040400406,2005-06-21,DET @ SAS,W,240,95,...,13,27,40,19,6,8,5,21,9.0,2004-05
12762,42004,1.610613e+09,SAS,San Antonio Spurs,0040400407,2005-06-23,SAS vs. DET,W,240,81,...,8,30,38,14,4,7,13,20,7.0,2004-05


In [10]:
# import os
# import re

# # 👉 Liste ici les dossiers des saisons à corriger
# season_dirs = [
#     "data/raw/boxscores/batches/2023-24", 
#     #"data/raw/boxscores/batches/2024-25"
# ]

# # ✅ Pattern pour détecter les fichiers batch à corriger
# pattern = re.compile(r"(boxscores_\w+_v3_batch_)(\d+)(\.csv)")

# # 🔁 Pour chaque saison et endpoint
# for season_dir in season_dirs:
#     for endpoint in os.listdir(season_dir):
#         endpoint_path = os.path.join(season_dir, endpoint)
#         if not os.path.isdir(endpoint_path):
#             continue
#         for filename in os.listdir(endpoint_path):
#             match = pattern.match(filename)
#             if match:
#                 prefix, number, suffix = match.groups()
#                 new_number = f"{int(number):03d}"
#                 new_name = f"{prefix}{new_number}{suffix}"
#                 old_path = os.path.join(endpoint_path, filename)
#                 new_path = os.path.join(endpoint_path, new_name)
#                 if old_path != new_path:
#                     os.rename(old_path, new_path)
#                     print(f"✅ Renamed: {filename} ➜ {new_name}")


# 3. Scraping des boxscores V3


In [11]:
#latest batches run_timestamp to complete the boxscores

#latest_batches_run_directory = "2025-06-03_14-23-24"
latest_batches_run_directory = run_timestamp

for season in seasons:
    print(f"--- Traitement de la saison {season} ---")
    season_df = games_df[games_df['SEASON'] == season]
    season_output_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season)
    scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

--- Traitement de la saison 2004-05 ---
[DEBUG] Found 52 batch files for endpoint 'traditional' in season 2004-05
[DEBUG] Found 52 batch files for endpoint 'advanced' in season 2004-05
[DEBUG] Checking last batch file for endpoint 'advanced': /home/ju/Documents/Dev/NBA_Predictor/data/legacy/raw/boxscores/batches/2004-05/advanced/boxscores_advanced_v3_batch_052.csv
[INFO] Resuming from gameId 0040400407 in season 2004-05
[DEBUG] Found 52 batch files for endpoint 'fourfactors' in season 2004-05
[DEBUG] Found 52 batch files for endpoint 'misc' in season 2004-05
[DEBUG] Found 52 batch files for endpoint 'scoring' in season 2004-05
[DEBUG] Found 52 batch files for endpoint 'usage' in season 2004-05
--------- 0 GAME_ID to scrap for season 2004-05 ---------
--- Traitement de la saison 2003-04 ---
[DEBUG] Found 50 batch files for endpoint 'traditional' in season 2003-04
[DEBUG] Found 50 batch files for endpoint 'advanced' in season 2003-04
[DEBUG] Checking last batch file for endpoint 'advance

In [12]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-11-05 11:47:33.677133
Total time:  10:54:44.353053


In [13]:
print("\n✅ Scraping historique V3 terminé")



✅ Scraping historique V3 terminé
